# TAR mission-planning training (Colab)

Trains MaskablePPO for offline mapper/relay mission planning.

**Runtime → Change runtime type → GPU** is recommended.

A short run proves the pipeline. Substantial learning needs much larger `TIMESTEPS` (e.g. 200k–1M+) and real eval against baselines.

## 1. Get the repo into Colab

Pick **one** option.

In [ ]:
# Option A: clone from GitHub (edit URL/branch if needed)
!git clone https://github.com/YOUR_ORG/T1_Software_Swarm.git
%cd T1_Software_Swarm

In [ ]:
# Option B: upload a zip of the repo, then:
# from google.colab import files
# uploaded = files.upload()  # choose T1_Software_Swarm.zip
# !unzip -q T1_Software_Swarm.zip
# %cd T1_Software_Swarm

## 2. Install planning dependencies

In [ ]:
import sys
print(sys.version)

!python -m pip -q install -r requirements-planning.txt
!python -m pip -q install -e ".[planning]"

import torch
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## 3. Train

- Start with `TIMESTEPS = 10_000` to verify Colab works.
- For a real attempt, try `200_000`–`1_000_000` with curriculum on.
- This does **not** guarantee beating the static/search baselines.

In [ ]:
from pathlib import Path
from tar_swarm.mission_planning.scenarios import one_mapper_small
from tar_swarm.mission_planning.train import train_maskable_ppo

TIMESTEPS = 10_000   # raise for real training, e.g. 200_000
N_ENVS = 4
SEED = 0
USE_CURRICULUM = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

out = Path("experiments/results/colab_plan_train")
summary = train_maskable_ppo(
    one_mapper_small(),
    out,
    total_timesteps=TIMESTEPS,
    n_envs=N_ENVS,
    seed=SEED,
    use_curriculum=USE_CURRICULUM,
    device=DEVICE,
)
summary

## 4. Evaluate vs baselines

In [ ]:
from tar_swarm.mission_planning.evaluate import evaluate_checkpoint
from tar_swarm.mission_planning.api import plan_mission

ckpt = summary["final_model"]
eval_result = evaluate_checkpoint(
    one_mapper_small(),
    ckpt,
    output_path=out / "eval.json",
)
eval_result

In [ ]:
# Explicit baseline (not attributed to RL)
baseline = plan_mission(one_mapper_small(), checkpoint=None, fallback="static")
print("baseline success:", baseline.success, baseline.source)

rl = plan_mission(one_mapper_small(), checkpoint=ckpt)
print("rl success:", rl.success, rl.source)
if rl.metrics:
    print({k: rl.metrics.get(k) for k in [
        "coverage_fraction", "disconnected_mapper_s",
        "relay_travel_m", "mission_duration_s", "success"
    ]})

## 5. Download checkpoint

In [ ]:
from google.colab import files
import shutil

bundle = out / "checkpoint_bundle"
bundle.mkdir(parents=True, exist_ok=True)
for name in ["final_model.zip", "final_model.meta.json"]:
    src = out / "checkpoints" / name
    if src.exists():
        shutil.copy(src, bundle / name)

zip_path = shutil.make_archive(str(out / "checkpoint_bundle"), "zip", bundle)
files.download(zip_path)
print("downloaded", zip_path)

## Resume later

```python
summary = train_maskable_ppo(
    one_mapper_small(),
    Path("experiments/results/colab_plan_train_resume"),
    total_timesteps=200_000,
    n_envs=4,
    resume_path="experiments/results/colab_plan_train/checkpoints/final_model.zip",
    use_curriculum=True,
    device=DEVICE,
)
```